# Forecasting Feature Engineering — Gated by Category/Product Variation

Rebuilds engineered features from `forecasting_daraz_daily_clean.csv` rather than trusting
its existing lag/rolling columns, since those were computed without the flat-price and
category-variation checks below.

Two output tracks:
- **Per-product** time-series features, only for products that (a) belong to a selected
  category and (b) show real price variation and (c) are still active near the end of the
  window.
- **Per-category** aggregate (median-based) series, for every category not cleared for
  per-product modeling — kept as a fallback rather than dropped.

Discount features are computed once, on the full dataset, independent of category selection.

In [ ]:
import pandas as pd
import numpy as np

DAILY_CSV = "output/forecasting_daraz_daily_clean.csv"
df = pd.read_csv(DAILY_CSV, parse_dates=["obs_date"])
df = df.sort_values(["url", "obs_date"]).reset_index(drop=True)
df.shape

## 1. EDA — per-product and per-category variation

Same check used to decide tea was the only broadly-modelable category: coefficient of
variation (CV) of `unit_price` per product, rolled up to a flat-vs-varying % per category.

In [ ]:
def product_variation_stats(frame):
    g = frame.groupby("url")["unit_price"]
    stats = g.agg(["count", "mean", "std"])
    stats["cv"] = (stats["std"] / stats["mean"]).fillna(0)
    stats["flat"] = stats["cv"] == 0
    stats["category"] = frame.groupby("url")["category"].first()
    return stats

product_stats = product_variation_stats(df)

category_summary = product_stats.groupby("category").agg(
    n_products=("flat", "size"),
    pct_flat=("flat", lambda x: round(x.mean() * 100, 1)),
    median_cv=("cv", "median"),
    median_obs=("count", "median"),
).sort_values("pct_flat")

category_summary

## 2. Category selection

Edit `SELECTED_CATEGORIES` to control which categories get a per-product time-series
pipeline. Everything else falls through to the per-category aggregate track in section 5.

Start with `["tea"]` — the only category where the median product actually shows price
movement. Add `"oil"` / `"rice"` once you're ready to accept a majority-flat product pool.

In [ ]:
SELECTED_CATEGORIES = ["rice","ghee" ,"tea"]

## 3. Product gating within selected categories

A product is eligible for per-product modeling only if:
- it belongs to a `SELECTED_CATEGORIES` entry,
- it shows nonzero price variation (`cv > 0` — flat products have nothing to predict),
- it was still observed recently, i.e. within `RECENCY_WINDOW_DAYS` of the dataset's last
  date. A product that stopped appearing partway through the window is likely delisted or
  out of stock and shouldn't anchor a forward-looking per-product forecast.

`RECENCY_WINDOW_DAYS` is deliberately a window, not a fixed calendar day, since "day 30"
keeps moving as more data is collected.

In [ ]:
RECENCY_WINDOW_DAYS = 3

last_date = df["obs_date"].max()
recency_cutoff = last_date - pd.Timedelta(days=RECENCY_WINDOW_DAYS)

last_seen = df.groupby("url")["obs_date"].max()
product_stats["last_seen"] = last_seen
product_stats["is_recent"] = product_stats["last_seen"] >= recency_cutoff

eligible_mask = (
    product_stats["category"].isin(SELECTED_CATEGORIES)
    & (product_stats["cv"] > 0)
    & product_stats["is_recent"]
)
eligible_products = product_stats[eligible_mask].index

print(f"{len(eligible_products)} / {product_stats['category'].isin(SELECTED_CATEGORIES).sum()} "
      f"selected-category products eligible for per-product modeling")

In [ ]:
cut_by_recency = product_stats[
    product_stats["category"].isin(SELECTED_CATEGORIES)
    & (product_stats["cv"] > 0)
    & (~product_stats["is_recent"])
]
cut_by_recency[["cv", "count", "last_seen"]].sort_values("last_seen")

## 4. Per-product feature engineering (eligible products only)

Recomputed fresh rather than reused from the source file. `_14` windows are included but
flagged separately — most products won't have 14 daily observations yet, so treat that
column as provisional until obs/product grows.

In [ ]:
per_product = df[df["url"].isin(eligible_products)].copy()
per_product = per_product.sort_values(["url", "obs_date"])

g = per_product.groupby("url")["unit_price"]

per_product["price_lag_1"] = g.shift(1)
per_product["price_lag_2"] = g.shift(2)
per_product["price_lag_7"] = g.shift(7)
per_product["price_lag_14"] = g.shift(14)  # provisional — most products won't clear this yet

per_product["rolling_mean_7"] = g.transform(lambda s: s.rolling(7, min_periods=1).mean())
per_product["rolling_std_7"] = g.transform(lambda s: s.rolling(7, min_periods=1).std())
per_product["rolling_min_7"] = g.transform(lambda s: s.rolling(7, min_periods=1).min())
per_product["rolling_max_7"] = g.transform(lambda s: s.rolling(7, min_periods=1).max())
per_product["rolling_mean_14"] = g.transform(lambda s: s.rolling(14, min_periods=1).mean())  # provisional

per_product["price_change"] = g.diff()
per_product["price_change_pct"] = g.pct_change()

per_product[["url", "obs_date", "unit_price", "price_lag_1", "price_lag_7",
             "rolling_mean_7", "rolling_std_7", "price_change_pct"]].head(10)

## 5. Per-category aggregate track (fallback for non-selected / gated-out categories)

Median-based series per category-day, so nothing gets dropped outright — categories that
didn't clear the per-product gate still contribute a usable signal at the aggregate level.

In [ ]:
category_daily = (
    df.groupby(["category", "obs_date"])["unit_price"]
    .median()
    .reset_index()
    .sort_values(["category", "obs_date"])
)

cg = category_daily.groupby("category")["unit_price"]
category_daily["cat_price_lag_1"] = cg.shift(1)
category_daily["cat_price_lag_7"] = cg.shift(7)
category_daily["cat_rolling_mean_7"] = cg.transform(lambda s: s.rolling(7, min_periods=1).mean())
category_daily["cat_price_change_pct"] = cg.pct_change()

category_daily.head(10)

## 6. Discount features — full dataset, independent of category selection

Computed across every category/product regardless of `SELECTED_CATEGORIES`, since the
discount-transition pattern (deepening discounts, `no_discount -> discount` price drops) is
a structural Daraz behavior, not something specific to which category is being modeled.

In [ ]:
df["has_discount"] = df["is_discounted"].astype(int)
df["discount_rate"] = df["discount_pct"].fillna(0)

drg = df.sort_values(["url", "obs_date"]).groupby("url")["has_discount"]
df["rolling_discount_rate"] = drg.transform(lambda s: s.rolling(7, min_periods=1).mean())

df[["url", "obs_date", "category", "is_discounted", "discount_pct",
    "rolling_discount_rate"]].head(10)

## 7. Auxiliary rating/review features (not primary predictors yet)

Kept as exploratory columns only — coverage is ~61-62%, and correlation with price hasn't
been validated. Included here so they're available once/if that check is done, not dropped
from the pipeline.

In [ ]:
df["has_rating"] = df["rating"].notna().astype(int)
df["has_review"] = df["review_count"].notna().astype(int)
df["log_review_count"] = np.log1p(df["review_count"].fillna(0))

## 8. Save outputs

In [ ]:
per_product.to_csv("forecast_data/selected_categories_per_product_features.csv", index=False)
category_daily.to_csv("forecast_data/category_daily_aggregate_features.csv", index=False)
df.to_csv("forecast_data/daraz_daily_with_discount_features.csv", index=False)

print("per_product:", per_product.shape)
print("category_daily:", category_daily.shape)
print("full with discount features:", df.shape)

In [ ]:
category_daily.groupby("category")["cat_price_change_pct"].agg(
    n_days="count",
    mean="mean",
    std="std",
    median="median",
    pct_nonzero=lambda s: (s != 0).mean() * 100
).round(4)